In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.impute import SimpleImputer
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


In [ ]:
df=pd.read_csv(r"C:\Users\Lenovo\Desktop\Final Project\bank_credit_dataset.csv")
df.head()

Data Cleanning

In [ ]:
null_ratio = df.isnull().mean()
null_ratio

we are finding null ratio for each column and comparing them with 0.7 ,if more that 70% of column is NaN ,we dont need them,they can affect model negatively

In [ ]:
high_null_cols = null_ratio[null_ratio >0.7].index.tolist()
print(f'Number of cols that will be dropped: {len(high_null_cols)}')

df_clean = df.drop(columns=high_null_cols)

df_clean

In [ ]:
numeric_df =df.select_dtypes(include=[np.number]).fillna(0)
numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)

selector = VarianceThreshold(threshold=0.01)
selector.fit(numeric_df)

kept_cols = numeric_df.columns[selector.get_support()].tolist()
removed = [c for c in numeric_df.columns if c not in kept_cols]

print(f'Dropped columns: len{removed}')
print(f'Kept columns : {len(kept_cols)}')

df_var = df_clean.drop(columns=removed,errors='ignore')


I used Variance Threshold method to filter and analyse columns due to threshold=0.01,if threshold< 0.01 ,it means we dont need that column

If we would use these columns instead of dropping them ,what would happen?:

    1. Model would work slowly than now
    
    2.Overfitting risk cuz of that kind of unuseful columns

Correlation filter

In [ ]:
numeric_df = df_var.select_dtypes(include=[np.number])
string_df  = df_var.select_dtypes(exclude=[np.number])

corr_matrix = numeric_df.drop(columns=['target'], errors='ignore').corr().abs()

mask      = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
upper_tri = corr_matrix.where(mask)

to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]

numeric_df_cleaned = numeric_df.drop(columns=to_drop)

print(f'Dropped numeric columns: {len(to_drop)}')
print(f'Kept numeric columns: {len(numeric_df_cleaned.columns)}')

LightGBM ilə feature importance 

In [ ]:
X = numeric_df_cleaned.drop(columns=['target', 'sk_id_curr', 'index'], errors='ignore')
y = df_var['target']


X = X.fillna(X.median())

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(X, y)

In [ ]:
importance_df = pd.DataFrame({
    'feature'   : X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

top_60_numeric = importance_df.head(60)['feature'].tolist()

importance_df.head(20).plot(
    x='feature', y='importance', kind='barh', figsize=(10, 8)
)
plt.title('Top 20 most important feature')
plt.tight_layout()
plt.show()


In [ ]:

cat_null_ratio = string_df.isnull().mean()
cat_keep = cat_null_ratio[cat_null_ratio <= 0.7].index.tolist()

df_final = pd.concat([
    numeric_df_cleaned[top_60_numeric], 
    string_df[cat_keep],             
    df_var[['target']]    
], axis=1)

In [ ]:
df_final.info()
string_cols = df.select_dtypes(include=['object'])
string_cols

In [ ]:
pd.set_option('display.max_rows',None)

In [ ]:
for col in string_cols:
    print(f"\n{'='*50}")
    print(f"Sütun: {col}")
    print(f"Unikal sayı: {df_final[col].nunique()}")
    print(f"Null faizi: {df_final[col].isnull().mean():.2%}")
    print(f"Dəyərlər: {df_final[col].value_counts().head(5).to_dict()}")

In [ ]:
summary = []
for col in df_final.select_dtypes(include=['object']).columns:
    summary.append({
        'column'   : col,
        'nunique'  : df_final[col].nunique(),
        'null_pct' : f"{df_final[col].isnull().mean():.1%}",
        'top_vals' : list(df_final[col].value_counts().head(3).index)
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

In [ ]:
# dropping high percentage of NaN
high_null_cat = ['housetype_mode','wallsmaterial_mode',
                 'fondkapremont_mode','emergencystate_mode']
df_final=df_final.drop(columns=high_null_cat)


In [ ]:
#There are XNA,we will make them NaN
df_final['code_gender']=df_final['code_gender'].replace('XNA',np.nan)

In [ ]:
df_final['organization_type']=df_final['organization_type'].replace('XNA',np.nan)

In [ ]:
df_final['age']=df_final['days_birth'] / -365
df_final.drop(columns='days_birth',inplace=True)
df_final.head()

In [ ]:
df_final['age']=df_final['age'].astype(int)


In [ ]:
df_final.to_pickle(r'C:\Users\Lenovo\Desktop\Home-Credit-Risk-Management\Datas\cleaned_data.pkl')

we can see that our target is inbalance (92% is Paid their credit ,and 8% is not paid their credit)